### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...


merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp (this one)

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping


Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [6]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 158984], edge_label=[158984])
Edge Index: tensor([[   0,    0,    0,  ..., 2063, 2063, 2063],
        [1102, 1186,  670,  ..., 2835,  742, 2969]])


#### Evaluate User Diversity Preference Scale

In [7]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:38<00:00, 53.48it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.511464,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.159621,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.424931,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545


In [8]:
encoded_train_df_with_dps = encoded_train_df.merge(user_dps_df, on="userID", how="left")
encoded_train_df_with_dps.head(2)

,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,...,directorID_idx,genre_idx,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,1588,2.0,29,10,2006,23,16,39,2006-10-29 23:16:39,...,2916,"[1, 2, 5, 16, 0, 0, 0, 0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.511464,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.159621,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.424931,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545
1,0,366,2.0,29,10,2006,23,16,42,2006-10-29 23:16:42,...,1882,"[1, 5, 6, 17, 0, 0, 0, 0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.511464,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.159621,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.424931,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545


#### Prepare prediction pool for inference/testing

In [9]:
# NOTE: Prepare prediction pool to evaluate the model
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2064(users) * 500(items) = 1032000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1031995,2063,447,0,"[15727, 2894, 10573, 253, 3728]",62,2985,"[8, 0, 0, 0, 0, 0, 0, 0]"
1031996,2063,3601,0,"[11716, 4113, 7140, 7336, 6954]",63,2854,"[1, 18, 0, 0, 0, 0, 0, 0]"
1031997,2063,7982,0,"[6627, 283, 3901, 11769, 14462]",63,1419,"[11, 0, 0, 0, 0, 0, 0, 0]"
1031998,2063,5969,0,"[11666, 10099, 4687, 6007, 6501]",63,3441,"[8, 15, 0, 0, 0, 0, 0, 0]"
1031999,2063,5025,0,"[3962, 7760, 5111, 6372, 4643]",62,558,"[6, 11, 14, 17, 0, 0, 0, 0]"


### Prepare DataLoader

In [10]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import UserItemPairDataset

BATCH_SIZE = 1024

train_dataset = UserItemPairDataset(encoded_train_df_with_dps)
valid_dataset = UserItemPairDataset(encoded_valid_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 358027
valid data count: 46916
test data count: 1032000


### Configure Model (LightningModule)

In [11]:
from lightning_models.bce.mtdp_gcn_cf_bce_rec import MTDPRecGCNCF

EMB_DIM = 32
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}
MT_WEIGHTS = {
    "rec_loss": 1.0,
    "dps_loss": 0.5,
}

model = MTDPRecGCNCF(
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,  # shape [2, num_edges]
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
    dps_weights=DPS_WEIGHTS,
    mt_weights=MT_WEIGHTS,
)


### Configure Trainer and Experiment

In [12]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "mtdp-gcn-bce-v1-exp"
RUN_NAME = "test8-2"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_f1",
    monitor_mode="max",
    hyper_param_str=f"emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}-batch_size={BATCH_SIZE}-epochs={EPOCHS}",
)

In [13]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    deterministic=True,
    callbacks=trainer_callbacks,
    accelerator='cpu',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


### Train Model

In [14]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/experiments/test_checkpoints/mtdp-gcn-bce-v1-exp exists and is not empty.

  | Name        | Type            | Params | Mode 
--------------------------------------------------------
0 | gcn_model   | GraphConvModule | 376 K  | train
1 | dps_module  | DPSPredictor    | 132    | train
2 | dps_loss_fn | DPSLoss         | 0      | train
--------------------------------------------------------
376 K     Trainable params
0         Non-trainable params
376 K     Total params
1.505     Total estimated model params size (MB)
50        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 1750: 'val_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_f1 did not improve in the last 5 records. Best score: 0.561. Signaling Trainer to stop.
Epoch 5, global step 2100: 'val_f1' was not in top 1


🏃 View run test8-2 at: http://140.112.106.216:3683/#/experiments/4/runs/01dee4368e5147bf954f53fbe8f90c0f
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/4


### Inference

In [15]:
# NOTE: the inference model MUST be the same as the training model
best_model_experiment_name = "mtdp-gcn-bce-v1-exp"
best_model_checkpoint_path = "test8-2-emb_dim=32-num_layers=3-lr=0.001-batch_size=1024-epochs=50-best-checkpoint-epoch=00-val_f1=0.56.ckpt"
best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"

model = MTDPRecGCNCF.load_from_checkpoint(
    checkpoint_path=best_model_path,
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    graph_data=train_graph,
    dim_id=EMB_DIM,
    num_layers=NUM_LAYERS,
    concat=True,
    lr=LR,
    dps_weights=DPS_WEIGHTS,
    mt_weights=MT_WEIGHTS,
)


In [16]:
# start inference
trainer.test(model=model, dataloaders=test_loader)

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │   0.032889533787965775    │
│          test_f1          │    0.05752182379364967    │
│         test_loss         │        583180.6875        │
│         test_prec         │    0.02961285598576069    │
│         test_rec          │    0.9997045993804932     │
└───────────────────────────┴───────────────────────────┘

🏃 View run test8-2 at: http://140.112.106.216:3683/#/experiments/4/runs/01dee4368e5147bf954f53fbe8f90c0f
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/4


[{'test_loss': 583180.6875,
  'test_acc': 0.032889533787965775,
  'test_prec': 0.02961285598576069,
  'test_rec': 0.9997045993804932,
  'test_f1': 0.05752182379364967}]

In [17]:
model.test_results

{'user': tensor([   0,    0,    0,  ..., 2063, 2063, 2063]),
 'item': tensor([7977, 4839,  979,  ..., 7982, 5969, 5025]),
 'score': tensor([1163490.6250,   62959.1484,  889536.7500,  ...,   13371.6875,
           60982.5312,   75463.3047]),
 'label': tensor([1., 0., 1.,  ..., 0., 0., 0.]),
 'metric': {'test_loss': 583180.6875,
  'test_acc': 0.03288953488372093,
  'test_prec': 0.029612855928891034,
  'test_rec': 0.9997045887218539,
  'test_f1': 0.0575218230510344},
 'user_emb': tensor([[-1.5414e+02, -4.3216e+00, -5.8891e+01,  ...,  3.4689e+03,
           9.1712e+03,  1.3180e+04],
         [-7.3178e+01, -2.0647e+00, -2.7991e+01,  ...,  1.6451e+03,
           4.3559e+03,  6.2471e+03],
         [-1.3190e+01, -3.6402e-01, -5.0302e+00,  ...,  2.9589e+02,
           7.8472e+02,  1.1273e+03],
         ...,
         [-4.2754e+02, -1.2168e+01, -1.6366e+02,  ...,  9.6288e+03,
           2.5431e+04,  3.6559e+04],
         [-7.7221e+02, -2.2635e+01, -2.9712e+02,  ...,  1.7420e+04,
           4.5933

In [18]:
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)
eval_df

,user,rec_items,gt_items
0,75,"[2571, 2959, 318, 4993, 7153, 5952, 356, 4306,...","[45722, 1233, 110, 2959, 2571]"
1,78,"[4306, 1221, 1732, 46578, 3052, 1225, 1090, 29...","[4119, 6993, 8400, 50872]"
2,127,"[2959, 2028, 4878, 1704, 1197, 1246, 48780, 17...","[45726, 6958]"
3,170,"[296, 260, 50, 1136, 1291, 1265, 44191, 1222, ...","[1222, 3949, 4011, 2542, 8874, 44191, 45728, 3..."
4,175,"[1580, 7147, 48394, 1721, 33493, 5995, 5902, 4...","[1913, 1419, 4927, 50068, 7700, 1757, 5300, 51..."
...,...,...,...
2059,71497,"[1089, 1291, 1214, 5418, 1682, 5989, 7147, 172...","[1608, 3175, 2006, 5418, 1909, 5989, 4246, 549..."
2060,71509,"[858, 1270, 2329, 1704, 6711, 1961, 1617, 750,...","[2019, 2238, 6783, 1231, 8914, 53887, 2010, 10..."
2061,71525,"[50, 858, 6874, 32587, 7438, 1193, 1265, 541, ...","[49530, 47099, 49278, 7147, 51575, 48304]"
2062,71529,"[5952, 3996, 1704, 1, 293, 1682, 46578, 1784, ...","[3996, 5952, 786, 1917, 2355, 1682]"


In [19]:
eval_score_df = evaluator.evaluate(eval_df, K=5)
eval_score_df = evaluator.evaluate(eval_score_df, K=10)
eval_score_df = evaluator.evaluate(eval_score_df, K=20)
eval_score_df.describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.340149,0.091630,0.177035,0.399344,0.170645,0.175484,0.438168,0.283106,0.157873
std,20797.975208,0.371210,0.156041,0.219892,0.325500,0.213779,0.179782,0.278892,0.264350,0.142844
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.270238,0.076923,0.050000
50%,35054.000000,0.386853,0.015625,0.200000,0.414437,0.100000,0.100000,0.446169,0.214286,0.150000
75%,53331.000000,0.630930,0.125000,0.200000,0.630930,0.250000,0.300000,0.635519,0.428571,0.250000
max,71534.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,0.800000
